# Day 049 — Exercise 1: cross_validate_model

**What you'll build:** `cross_validate_model(model, X, y, cv=5, scoring='r2') -> dict` — run k-fold cross-validation and return per-fold scores plus summary statistics (mean, std, min, max).

**Why it matters:** A single train/test split is sensitive to which 80% you happened to train on. With k-fold CV, the model is trained and evaluated k times on different subsets — the mean score is a far more reliable estimate of real-world performance.

## Provided: Setup + Data Generators

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                              accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')


def make_regression_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Numeric-only housing dataset (area, bedrooms, age → price)."""
    rng = np.random.default_rng(seed)
    area     = rng.uniform(500, 3000, n).round(0)
    bedrooms = rng.integers(1, 6, n)
    age      = rng.uniform(0, 50, n).round(1)
    price    = (area * 150 + bedrooms * 10_000 - age * 1_000
                + rng.standard_normal(n) * 10_000).round(-2)
    return pd.DataFrame({'area': area.astype(int), 'bedrooms': bedrooms,
                         'age': age, 'price': price.astype(int)})


def make_classification_data(n: int = 200, seed: int = 42) -> pd.DataFrame:
    """Student exam dataset: hours_studied + hours_sleep → passed (0/1)."""
    rng          = np.random.default_rng(seed)
    hours_studied = rng.uniform(0, 10, n).round(1)
    hours_sleep   = rng.uniform(4, 10, n).round(1)
    noise         = rng.standard_normal(n)
    score         = 1.5 * hours_studied + 0.5 * hours_sleep + noise
    passed        = (score > 9.0).astype(int)
    return pd.DataFrame({'hours_studied': hours_studied,
                         'hours_sleep':   hours_sleep,
                         'passed':        passed})

## Your Implementation

In [ ]:
def cross_validate_model(model, X: pd.DataFrame, y: pd.Series,
                          cv: int = 5,
                          scoring: str = 'r2') -> dict:
    """
    K-fold cross-validation.

    Args:
        model:   unfitted sklearn estimator
        X, y:    full feature matrix and target (unsplit)
        cv:      number of folds (default 5)
        scoring: sklearn scoring string ('r2', 'accuracy', etc.)
    Returns:
        dict with keys: scores, mean, std, min, max, cv_folds, scoring
    """
    # TODO: kf     = KFold(n_splits=cv, shuffle=True, random_state=42)
    # TODO: scores = cross_val_score(model, X, y, cv=kf, scoring=scoring)
    # TODO: return {
    #     'scores':   scores,
    #     'mean':     round(float(scores.mean()), 4),
    #     'std':      round(float(scores.std()),  4),
    #     'min':      round(float(scores.min()),  4),
    #     'max':      round(float(scores.max()),  4),
    #     'cv_folds': cv,
    #     'scoring':  scoring,
    # }
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df = make_regression_data(200)
    X  = df.drop(columns=['price'])
    y  = df['price']

    # Check 1: defined, returns dict
    try:
        assert 'cross_validate_model' in globals()
        result = cross_validate_model(LinearRegression(), X, y)
        assert isinstance(result, dict), \
            f'expected dict, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: cross_validate_model returns dict')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: all 7 keys present
    try:
        for k in ('scores', 'mean', 'std', 'min', 'max', 'cv_folds', 'scoring'):
            assert k in result, f'missing key: {k!r}'
        passed += 1; print('\u2705 Check 2: all 7 keys present')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: len(scores) == cv (default 5)
    try:
        assert len(result['scores']) == 5, \
            f'expected 5 scores (one per fold), got {len(result["scores"])}'
        assert result['cv_folds'] == 5
        passed += 1; print(f'\u2705 Check 3: {len(result["scores"])} scores (5 folds)')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: mean R² > 0.9 on housing data
    try:
        assert result['mean'] > 0.9, \
            f'mean R\u00b2 should be > 0.9 on housing data, got {result["mean"]}'
        passed += 1; print(f'\u2705 Check 4: mean R\u00b2={result["mean"]} > 0.9')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: std < 0.1 (consistent across folds)
    try:
        assert result['std'] < 0.1, \
            f'std should be < 0.1 (consistent model), got {result["std"]}'
        passed += 1; print(f'\u2705 Check 5: std={result["std"]} < 0.1 (stable CV)')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def cross_validate_model(model, X: pd.DataFrame, y: pd.Series,
                          cv: int = 5,
                          scoring: str = 'r2') -> dict:
    """K-fold cross-validation returning per-fold scores and summary stats."""
    kf     = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring=scoring)
    return {
        'scores':   scores,
        'mean':     round(float(scores.mean()), 4),
        'std':      round(float(scores.std()),  4),
        'min':      round(float(scores.min()),  4),
        'max':      round(float(scores.max()),  4),
        'cv_folds': cv,
        'scoring':  scoring,
    }
```

</details>